# Baseline models

This notebook evaluates the benchmark forecasting models on the
cleaned train, validation and test splits. Simple statistical models are also fit directly here.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current available evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Pearson Correlation between predictions and target in cumulative log change space**
4. **Relative MAE versus Persistence**
5. **Persistence win rate**

Bootstrap evaluation metrics over the test datset are available and used by default.

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.
6. **ModernTCN** - multiple ablations have been trained in Colab. The
    best version checkpoint (based on validation loss) is called and used to 
    predict. It is the version which takes all OHLCV as input, adds a time
    of day temporal feature and flattens series into batch (so we dont mix
    across asset - huge reduction in parameter count - 121,138 params in total
    including 256 params added for the temporal feature).

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_evaluation_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

## Load the data and clean

In [3]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

#Set global Bootstrap params
BOOTSTRAP_N = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 42


train samples: 167
val samples: 20
test samples: 62
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume']
targets: ['close']


## Persistence

In [4]:
persistence = PersistenceBaseline.from_config(
    config
)

persistence.fit(
    train_split=train,
    val_split=val,
)

persistence_result = persistence.predict(
    split=test,
    batch_size=256,
)

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=train,
)


# Persistence predicts zero cumulative log change at every horizon,
# so its cumulative-log-change Pearson correlation or IC is undefined.
persistence_undefined_metrics = {
    "cumulative_log_change_pearson_correlation",
    "cumulative_log_change_cross_sectional_pearson_ic",
}

persistence_metric_names = [
    metric_name
    for metric_name in persistence_evaluator.available_metrics
    if metric_name not in persistence_undefined_metrics
]


persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_metric_names,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)


for metric_name in persistence_metric_names:
    metric_display = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000367,0.000352,0.000383,0.000367,0.000008
5,close,0.000785,0.000757,0.000815,0.000785,0.000015
15,close,0.001322,0.001276,0.001371,0.001322,0.000025
30,close,0.001838,0.001767,0.001914,0.001839,0.000038
60,close,0.002553,0.002428,0.002697,0.002554,0.000069


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.949704,0.911210,0.991452,0.949813,0.020469
5,close,2.041615,1.968282,2.119840,2.041683,0.038680
15,close,3.432314,3.312254,3.561779,3.432670,0.063754
30,close,4.775831,4.587843,4.975652,4.776736,0.099277
60,close,6.649479,6.324530,7.021874,6.651386,0.178045


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.000000,1.000000,1.000000,1.000000,0.000000
5,close,1.000000,1.000000,1.000000,1.000000,0.000000
15,close,1.000000,1.000000,1.000000,1.000000,0.000000
30,close,1.000000,1.000000,1.000000,1.000000,0.000000
60,close,1.000000,1.000000,1.000000,1.000000,0.000000


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.500000,0.500000,0.500000,0.500000,0.000000
5,close,0.500000,0.500000,0.500000,0.500000,0.000000
15,close,0.500000,0.500000,0.500000,0.500000,0.000000
30,close,0.500000,0.500000,0.500000,0.500000,0.000000
60,close,0.500000,0.500000,0.500000,0.500000,0.000000


## Mean

In [5]:
mean = MeanBaseline.from_config(
    config
)

mean.fit(
    train_split=train,
    val_split=val,
)

mean_result = mean.predict(
    split=test,
    batch_size=256,
)

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_display = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.001651,0.001582,0.001725,0.001651,0.000037
5,close,0.001787,0.001711,0.001867,0.001787,0.000040
15,close,0.002075,0.001984,0.002172,0.002075,0.000048
30,close,0.002429,0.002317,0.002550,0.002429,0.000060
60,close,0.003012,0.002851,0.003197,0.003013,0.000089


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.001426,-0.027209,0.032355,0.001558,0.015368
5,close,0.009109,-0.007818,0.027211,0.009177,0.008866
15,close,0.000604,-0.026585,0.026697,0.001032,0.013675
30,close,0.000925,-0.035819,0.035174,0.001521,0.018281
60,close,-0.005952,-0.064780,0.042979,-0.004699,0.028333


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,4.292427,4.111240,4.487903,4.293373,0.096311
5,close,4.646160,4.451149,4.856565,4.647267,0.104283
15,close,5.394403,5.158553,5.651376,5.395818,0.126212
30,close,6.318103,6.029741,6.632837,6.320128,0.156057
60,close,7.850683,7.438304,8.321093,7.853870,0.226875


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,4.477811,4.345123,4.616718,4.479067,0.069203
5,close,2.234747,2.185501,2.286766,2.235230,0.025720
15,close,1.553153,1.514623,1.594874,1.553365,0.020413
30,close,1.314323,1.287402,1.343726,1.314403,0.014433
60,close,1.171088,1.149794,1.192419,1.171135,0.010806


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.136179,0.132218,0.140506,0.136146,0.002087
5,close,0.250657,0.244856,0.256394,0.250589,0.002919
15,close,0.334547,0.327792,0.341339,0.334461,0.003459
30,close,0.377161,0.370388,0.383665,0.377076,0.003318
60,close,0.408607,0.401368,0.415745,0.408506,0.003709


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.022039,0.007991,0.035935,0.022031,0.007115
5,close,0.025570,0.012035,0.038997,0.025492,0.006908
15,close,0.026818,0.011433,0.041826,0.026733,0.007833
30,close,0.031292,0.011925,0.049914,0.031187,0.009686
60,close,0.031328,0.009459,0.051970,0.031174,0.010830


## ARIMA

In [9]:
arima = ArimaBaseline.from_config(
    config,
    fit_mode="simple",
    optim_method="powell",
)

arima.fit(
    train_split=train,
    val_split=val,
)

arima_result = arima.predict(
    split=test,
    batch_size=32,
)

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_display = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 93 ARIMA models using fit_mode='simple'...
  fitted 25/93
  fitted 50/93
  fitted 75/93
Finished fitting ARIMA models.
Failed models: 0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000367,0.000352,0.000383,0.000367,0.000008
5,close,0.000785,0.000757,0.000815,0.000785,0.000015
15,close,0.001323,0.001277,0.001372,0.001323,0.000025
30,close,0.001840,0.001769,0.001916,0.001840,0.000038
60,close,0.002558,0.002432,0.002702,0.002559,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.031713,0.002859,0.055658,0.031802,0.013499
5,close,0.015962,0.000012,0.032305,0.015862,0.008151
15,close,-0.001287,-0.014811,0.011538,-0.001236,0.006723
30,close,0.000689,-0.013713,0.015156,0.000709,0.007445
60,close,-0.000526,-0.022011,0.020246,-0.000568,0.010791


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.951805,0.913396,0.993582,0.951912,0.020428
5,close,2.042834,1.969567,2.121091,2.042902,0.038696
15,close,3.435128,3.315008,3.564510,3.435481,0.063772
30,close,4.781092,4.593257,4.980287,4.781996,0.099431
60,close,6.664249,6.336887,7.038714,6.666158,0.179339


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.004039,1.002129,1.005971,1.004026,0.000979
5,close,0.998848,0.997995,0.999665,0.998850,0.000426
15,close,1.000198,0.999645,1.000763,1.000197,0.000286
30,close,1.000399,0.999675,1.001140,1.000400,0.000372
60,close,1.000953,0.999752,1.002126,1.000953,0.000607


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.458436,0.453132,0.463922,0.458446,0.002790
5,close,0.486272,0.480462,0.492054,0.486267,0.002941
15,close,0.489649,0.483506,0.495765,0.489678,0.003086
30,close,0.489731,0.481215,0.498079,0.489754,0.004316
60,close,0.485103,0.472730,0.497116,0.485104,0.006219


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.050016,0.041882,0.058192,0.050041,0.004214
5,close,0.020017,0.009892,0.029777,0.020042,0.005084
15,close,0.009296,-0.000641,0.019055,0.009259,0.005007
30,close,0.002837,-0.012381,0.018057,0.002825,0.007801
60,close,-0.001554,-0.023894,0.020535,-0.001526,0.011371


## VAR

In [6]:
var = VarBaseline.from_config(
    config,
    maxlags=15,
    ic="aic",
    trend="c",
)

var.fit(
    train_split=train,
    val_split=val,
)

var_result = var.predict(
    split=test,
    batch_size=256,
)

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_display = (
        var_metric_table
        .loc[
            var_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 1 VAR model(s) with maxlags=15, ic=aic...
  close: selected_lag=11, failed=False
Finished fitting VAR models.
Failed models: 0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000379,0.000364,0.000395,0.000379,0.000008
5,close,0.000799,0.000770,0.000830,0.000799,0.000015
15,close,0.001335,0.001288,0.001385,0.001335,0.000025
30,close,0.001848,0.001776,0.001924,0.001848,0.000038
60,close,0.002563,0.002437,0.002707,0.002563,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.010872,-0.004869,0.026517,0.010967,0.008019
5,close,0.015784,0.000930,0.031262,0.015887,0.007697
15,close,-0.002537,-0.018208,0.012744,-0.002487,0.007854
30,close,0.004379,-0.012439,0.020928,0.004453,0.008567
60,close,0.002263,-0.012588,0.017160,0.002322,0.007576


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.982359,0.944070,1.024233,0.982443,0.020322
5,close,2.076806,2.002656,2.156407,2.076869,0.039204
15,close,3.464684,3.342842,3.597083,3.465064,0.064807
30,close,4.800755,4.611415,5.002028,4.801647,0.100122
60,close,6.675028,6.347023,7.049401,6.676942,0.179296


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.039215,1.034092,1.044508,1.039190,0.002680
5,close,1.019413,1.014942,1.023997,1.019414,0.002308
15,close,1.008641,1.005642,1.011762,1.008662,0.001555
30,close,1.004279,1.001789,1.006703,1.004283,0.001254
60,close,1.003097,1.001223,1.005025,1.003100,0.000971


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.434530,0.429501,0.439546,0.434573,0.002584
5,close,0.469837,0.464981,0.474520,0.469835,0.002449
15,close,0.475889,0.470768,0.480904,0.475890,0.002577
30,close,0.482826,0.477335,0.488307,0.482834,0.002784
60,close,0.484953,0.477632,0.492155,0.484942,0.003711


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.027007,0.017798,0.036683,0.027003,0.004833
5,close,0.007813,-0.001282,0.017118,0.007844,0.004721
15,close,-0.007751,-0.016983,0.001789,-0.007779,0.004746
30,close,-0.002462,-0.012944,0.008271,-0.002493,0.005442
60,close,-0.001407,-0.014988,0.012777,-0.001410,0.007154


## GARCH

In [7]:
garch = GarchBaseline.from_config(
    config,
    mean="AR",
    return_scale=10000.0,
)

garch.fit(
    train_split=train,
    val_split=val,
)

garch_result = garch.predict(
    split=test,
    batch_size=256,
)

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_display = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 93 GARCH(1,1) models with mean='AR'...
  fitted 25/93
  fitted 50/93
  fitted 75/93
Finished fitting GARCH models.
Failed models: 0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000368,0.000353,0.000384,0.000368,0.000008
5,close,0.000785,0.000757,0.000816,0.000785,0.000015
15,close,0.001323,0.001277,0.001373,0.001323,0.000025
30,close,0.001841,0.001769,0.001917,0.001841,0.000038
60,close,0.002560,0.002433,0.002706,0.002561,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.031935,0.005737,0.055063,0.031912,0.012540
5,close,0.014379,-0.002126,0.031008,0.014254,0.008470
15,close,0.000941,-0.015292,0.015555,0.000997,0.007813
30,close,0.002778,-0.010144,0.015023,0.002865,0.006420
60,close,0.003407,-0.010374,0.017557,0.003569,0.007142


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.952390,0.913917,0.994147,0.952497,0.020432
5,close,2.042920,1.969624,2.121329,2.042993,0.038719
15,close,3.435790,3.315743,3.565393,3.436143,0.063774
30,close,4.782484,4.594722,4.982995,4.783394,0.099694
60,close,6.668193,6.338200,7.044449,6.670116,0.180179


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.005828,1.003290,1.008365,1.005813,0.001296
5,close,0.999117,0.998180,0.999993,0.999119,0.000466
15,close,1.000455,0.999625,1.001283,1.000452,0.000427
30,close,1.000854,0.999480,1.002257,1.000850,0.000706
60,close,1.001783,0.999338,1.004198,1.001772,0.001233


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.456903,0.451275,0.462653,0.456904,0.002926
5,close,0.485596,0.478777,0.492378,0.485561,0.003449
15,close,0.487458,0.478490,0.496458,0.487474,0.004594
30,close,0.487705,0.475254,0.500224,0.487707,0.006411
60,close,0.482064,0.464182,0.499740,0.482045,0.009051


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.045634,0.036621,0.054782,0.045639,0.004653
5,close,0.019591,0.011253,0.027769,0.019543,0.004227
15,close,0.013457,0.005118,0.021612,0.013452,0.004210
30,close,0.010475,0.000427,0.020606,0.010523,0.005190
60,close,0.008838,-0.005507,0.023354,0.008922,0.007366


In [ ]:
modern_tcn_checkpoint_path = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/checkpoints/modern_tcn/"
    "per_asset_cv_to_c/runs/um9rdmc6/"
    "best_checkpoint.pt"
).expanduser().resolve()

modern_tcn = ModernTCNBaseline.from_config(
    config
)

modern_tcn.load_checkpoint(
    checkpoint_path=modern_tcn_checkpoint_path,
    device="cpu",
)

modern_tcn_result = modern_tcn.predict(
    split=test,
    batch_size=8,
    num_workers=0,
)

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)

modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    metric_display = (
        modern_tcn_metric_table
        .loc[
            modern_tcn_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000369,0.000354,0.000385,0.000369,0.000008
5,close,0.000786,0.000758,0.000816,0.000786,0.000015
15,close,0.001323,0.001277,0.001372,0.001323,0.000024
30,close,0.001840,0.001769,0.001916,0.001841,0.000038
60,close,0.002560,0.002434,0.002706,0.002561,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.028538,0.007305,0.051882,0.028481,0.011285
5,close,0.005314,-0.012656,0.023555,0.005292,0.009243
15,close,0.018456,-0.000221,0.037467,0.018496,0.009630
30,close,0.011075,-0.011973,0.035384,0.011036,0.012055
60,close,-0.004687,-0.037022,0.025483,-0.004249,0.016009


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.957202,0.919001,0.998733,0.957311,0.020346
5,close,2.045127,1.971319,2.123394,2.045203,0.038837
15,close,3.435309,3.315108,3.564741,3.435649,0.063770
30,close,4.782403,4.594673,4.982570,4.783302,0.099260
60,close,6.667828,6.337933,7.043028,6.669757,0.179551


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.008515,1.006475,1.010584,1.008516,0.001047
5,close,1.000110,0.998663,1.001529,1.000110,0.000739
15,close,1.000217,0.999092,1.001383,1.000219,0.000586
30,close,1.001501,1.000004,1.003036,1.001505,0.000773
60,close,1.003249,1.001091,1.005410,1.003239,0.001101


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.450098,0.444073,0.456323,0.450099,0.003133
5,close,0.485135,0.479019,0.491201,0.485117,0.003104
15,close,0.493373,0.487654,0.499188,0.493393,0.002959
30,close,0.488499,0.482529,0.494295,0.488496,0.003038
60,close,0.479850,0.467440,0.491712,0.479824,0.006184


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.033897,0.023836,0.044016,0.033893,0.005145
5,close,0.007267,-0.002776,0.017708,0.007296,0.005229
15,close,0.014916,0.004632,0.025137,0.014874,0.005265
30,close,0.000663,-0.012175,0.013583,0.000605,0.006595
60,close,0.002033,-0.012817,0.016920,0.002004,0.007520
